# 第 15 天：因子标准化

> 来自《30 天因子研究计划》第 15 天  
> 主题：因子标准化  
> 必做：去极值  
> 选做：标准化  
> 目标产出：预处理流水线

---

## 0. 今天你要真正学会什么？

因子预处理是在问：


原始因子能不能直接拿去比较、合成和回测？


多数时候答案是：不能。

今天掌握：

1. 为什么要去极值。
2. 分位数去极值和 MAD 去极值。
3. Z-score 标准化。
4. 如何搭建预处理流水线。

---

## 1. 为什么要预处理？

原始因子常见问题：

- 极端值太大。
- 不同行业分布不同。
- 不同因子量纲不同。
- 合成多因子时无法直接相加。

预处理的常见顺序：


原始因子 → 去极值 → 中性化 → 标准化 → 因子合成


今天重点放在去极值和标准化。

---

## 2. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260706)


---

## 3. 构造带极端值的因子


In [ ]:
n = 500
df = pd.DataFrame({
    "ticker": [f"Stock_{i:03d}" for i in range(n)],
    "raw_factor": rng.normal(0, 1, size=n),
})

outlier_idx = rng.choice(n, size=10, replace=False)
df.loc[outlier_idx[:5], "raw_factor"] *= 12
df.loc[outlier_idx[5:], "raw_factor"] -= 15

df["raw_factor"].describe()


---

## 4. 分位数去极值


In [ ]:
def quantile_winsorize(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


df["factor_q_winsor"] = quantile_winsorize(df["raw_factor"], 0.01, 0.99)
df[["raw_factor", "factor_q_winsor"]].describe()


---

## 5. MAD 去极值

MAD 是中位数绝对偏差，更稳健。


In [ ]:
def mad_winsorize(s: pd.Series, n_mad: float = 3.0) -> pd.Series:
    median = s.median()
    mad = (s - median).abs().median()
    if mad == 0 or pd.isna(mad):
        return s.copy()
    lo = median - n_mad * 1.4826 * mad
    hi = median + n_mad * 1.4826 * mad
    return s.clip(lo, hi)


df["factor_mad_winsor"] = mad_winsorize(df["raw_factor"], 3)
df[["raw_factor", "factor_mad_winsor"]].describe()


---

## 6. Z-score 标准化


In [ ]:
def zscore(s: pd.Series) -> pd.Series:
    std = s.std()
    if std == 0 or pd.isna(std):
        return s * np.nan
    return (s - s.mean()) / std


df["factor_zscore"] = zscore(df["factor_mad_winsor"])
df["factor_zscore"].describe()


标准化后，因子大致满足：


均值约为 0
标准差约为 1


---

## 7. 可视化处理效果


In [ ]:
df[["raw_factor", "factor_mad_winsor", "factor_zscore"]].hist(bins=40, figsize=(10, 7))
plt.tight_layout()
plt.show()


---

## 8. 多日期因子预处理

真实因子需要每天在截面上处理，不能把所有日期混在一起。


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=30)
panel = []
for date in dates:
    temp = df[["ticker"]].copy()
    temp["date"] = date
    temp["raw_factor"] = rng.normal(0, 1, size=n)
    temp.loc[rng.choice(n, size=5, replace=False), "raw_factor"] *= 10
    panel.append(temp)
panel = pd.concat(panel, ignore_index=True)

panel.head()


---

## 9. 目标产出：预处理流水线


In [ ]:
def preprocess_factor(s: pd.Series, method: str = "mad") -> pd.Series:
    if method == "mad":
        cleaned = mad_winsorize(s)
    elif method == "quantile":
        cleaned = quantile_winsorize(s)
    else:
        raise ValueError("method must be 'mad' or 'quantile'")
    return zscore(cleaned)


def preprocess_factor_panel(
    data: pd.DataFrame,
    factor_col: str,
    date_col: str = "date",
    output_col: str = "factor_processed",
    method: str = "mad",
) -> pd.DataFrame:
    out = data.copy()
    out[output_col] = out.groupby(date_col)[factor_col].transform(
        lambda s: preprocess_factor(s, method=method)
    )
    return out


processed = preprocess_factor_panel(panel, "raw_factor", method="mad")
processed.head()


检查每天均值和标准差：


In [ ]:
check = processed.groupby("date")["factor_processed"].agg(["mean", "std"]).head()
check


---

## 10. 加入缺失值处理


In [ ]:
panel_with_nan = panel.copy()
panel_with_nan.loc[rng.choice(len(panel_with_nan), size=200, replace=False), "raw_factor"] = np.nan

processed_nan = preprocess_factor_panel(panel_with_nan, "raw_factor")
missing_check = processed_nan["factor_processed"].isna().mean()
missing_check


缺失值通常保留为缺失，不要随便填 0。  
因为 0 在标准化后代表“中性水平”，而缺失代表“不知道”。

---

## 11. 知识图谱


In [ ]:
mindmap
  root((因子标准化))
    去极值
      分位数
      MAD
      clip
    标准化
      zscore
      均值0
      标准差1
    截面处理
      每日分组
      不混合日期
    流水线
      原始因子
      清洗
      标准化
      输出


---

## 12. 作业

1. 比较 MAD 和分位数去极值的结果。
2. 把 MAD 倍数从 3 改成 2 和 5。
3. 检查处理后每天均值是否接近 0。
4. 思考为什么缺失值不能随便填 0。

---

## 13. 自测题

1. 去极值解决什么问题？  
   答案：减少极端值对排序、标准化和回测的影响。

2. Z-score 的公式是什么？  
   答案：`(x - mean) / std`。

3. 为什么要每天截面标准化？  
   答案：因子每天的分布和股票池可能不同。

---

## 14. 后续预告

第 16 天开始进入 Alpha101 导论和公式复现。  
前 15 天的因子构建、检验和预处理，是复现 Alpha 因子的底座。

---

## 15. 仅供学习的提醒

本文使用模拟数据解释因子预处理方法，不构成任何投资建议。

---

# 统一高质量增强模块

> 本增强模块用于把第 15 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：因子标准化
- 必做：去极值
- 选做：标准化
- 目标产出：预处理流水线

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

一个极端值能把均值和标准差全部带偏，让因子合成失真。预处理就是让数据先变得可比较、可合成。

这个例子背后的关键直觉是：

> 先让因子干净，再讨论因子有效。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


因子标准化
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 预处理流水线


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(115)
s = pd.Series(rng.normal(0, 1, 500))
s.iloc[:5] = [10, -12, 8, -9, 15]

def mad_winsorize(x, n_mad=3):
    med = x.median()
    mad = (x - med).abs().median()
    lo = med - 1.4826 * n_mad * mad
    hi = med + 1.4826 * n_mad * mad
    return x.clip(lo, hi)

cleaned = mad_winsorize(s)
z = (cleaned - cleaned.mean()) / cleaned.std()
print(pd.DataFrame({"raw": s, "cleaned": cleaned, "zscore": z}).describe().round(3))


## E. 产出验收标准

完成今天课程后，你的 `预处理流水线` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `因子标准化` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `预处理流水线` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `预处理流水线`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 15 天复盘：因子标准化

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
---

# 第 10-15 天深度加厚模块


    ## L. 为什么还要加厚这一课？

    这一课属于第 10-15 天的“工程化因子”部分：它不像 Alpha、Beta 那样只靠概念就能建立直觉，也不像 PE、ROE 那样有明确财务含义。它更依赖窗口、参数、预处理顺序和检验口径。

    所以学习 `因子标准化` 时，不能只停留在“知道公式”。你至少要完成三层理解：


    第一层：公式能写对
    第二层：参数变化后结果还能解释
    第三层：能放进统一因子流水线


    如果只学第一层，代码很快能写出来，但研究时很容易陷入“参数换一下结果就变”的困境。

    ## M. 更贴近真实研究的场景

    一个极端值可能让 z-score 全体失真。一个缺失值如果被填成 0，可能被误解为中性。预处理看似枯燥，却决定因子合成是否可靠。

    这类问题在真实研究中很常见：一个因子看似简单，但只要换股票池、换窗口、换持有期、换市场阶段，结果就会明显变化。成熟的研究方式不是逃避这种变化，而是把变化记录下来、解释出来。

    ## N. 参数敏感性实验

    下面这段代码是专门为本课补充的参数实验。它不追求复杂，而是训练一个习惯：

    > 不要只交一个因子结果，至少比较几组合理参数。


In [ ]:
    import numpy as np
import pandas as pd

rng = np.random.default_rng(215)
dates = pd.bdate_range("2024-01-02", periods=10)
rows = []
for d in dates:
    x = rng.normal(0,1,200)
    x[:3] = [12, -10, 15]
    rows.extend(zip([d]*200, [f"S{i:03d}" for i in range(200)], x))
df = pd.DataFrame(rows, columns=["date","ticker","raw_factor"])

def mad_winsor(s, k=3):
    med = s.median()
    mad = (s-med).abs().median()
    return s.clip(med-1.4826*k*mad, med+1.4826*k*mad)

def zscore(s):
    return (s-s.mean())/s.std()

df["processed"] = df.groupby("date")["raw_factor"].transform(lambda s: zscore(mad_winsor(s)))
print(df.groupby("date")["processed"].agg(["mean","std"]).round(4).head())
print(df[["raw_factor","processed"]].describe().round(3))


    ## O. 结果该怎么写进研究笔记？

    建议你用下面这个格式记录：


    因子名称：因子标准化

    1. 使用的数据：
       - 股票池：
       - 时间区间：
       - 价格 / 财务口径：

    2. 核心参数：
       - 主参数：
       - 对照参数：

    3. 因子方向：
       - 因子值越大代表：
       - 是否需要取负号：

    4. 检验结果：
       - Rank IC：
       - ICIR：
       - 分组收益：
       - 多空表现：

    5. 稳定性：
       - 参数变化后是否稳定：
       - 分阶段是否稳定：
       - 极端行情是否失效：

    6. 结论：
       - 是否进入因子库：
       - 还需要什么后续验证：


    ## P. 额外验收清单

    `预处理流水线` 如果要达到可复用标准，额外检查：

    1. 去极值和标准化必须按日期截面做。
2. 缺失值不要默认填 0。
3. 处理后每日均值约 0、标准差约 1。
4. 记录使用的是 MAD 还是分位数去极值。

    ## Q. 更深入的常见误区

    ### 误区 1：参数越多越高级

    参数多不代表研究深，很多时候只是过拟合空间更大。真正高级的是解释参数为什么合理，并证明它在相邻参数下仍然不崩。

    ### 误区 2：只看全样本平均

    全样本平均可能掩盖阶段失效。至少要分年度、分市场状态、分股票池看一次。

    ### 误区 3：把预处理当成机械步骤

    去极值、标准化、中性化会改变因子含义。每加一步，都要知道自己剥离了什么，也可能损失了什么。

    ### 误区 4：忽略交易可行性

    技术、波动率、流动性类因子往往换手更高，交易成本可能非常关键。纸面有效不等于可交易。

    ## R. 加厚作业

    1. 把本课主参数上下各调整一次，记录结果变化。
    2. 把未来收益标签从 20 日改成 5 日和 60 日，观察结论是否变。
    3. 随机删除 10% 股票样本，检查结果是否稳定。
    4. 把因子取反，确认分组结果是否镜像变化。
    5. 写一段 200 字研究结论，必须同时包含“支持证据”和“风险提示”。

    ## S. 一句话升级结论

    `因子标准化` 的高质量学习标准不是“会算”，而是：

    > 会定义、会检验、会解释参数变化，也知道它在真实交易里可能被什么击穿。
